# Manager notebook -- yolo11 waste-detection fine-tune

Thin Colab orchestrator over the `trashmonkey` package: zero logic in
cells, only imports and calls (manager-notebook architecture). Designed to
survive "Runtime > Run all" on a fresh Colab session. Fill in the credential
constants in the Init cell before running on Colab.

## 1. Init

Constants, Colab detection, Drive mount, and a preflight that collects every
problem before a single assert. `torch`/`ultralytics` are required on Colab
only (they ship with the runtime); locally their absence is a warning.

In [ ]:
# Init: config constants, Colab detection, Drive mount, credential presence.
# Package-import validation happens AFTER the install cell (deps are not present
# on a fresh Colab until then), so this cell only checks the environment.
import os
import sys
from pathlib import Path

GITHUB_USERNAME = "public"  # fill in: GitHub account with read access to the repo
GITHUB_TOKEN = "public"  # fill in: personal access token (repo read scope)
REPO_URL = "https://github.com/tr4m0ryp/TrashMonkey.git"  # fill in: e.g. https://github.com/<user>/TrashMonkey.git

# Kaggle API token (kaggle.com -> Settings -> API -> Create New Token). Needed
# the first time the dataset is built (to fetch the Kaggle source datasets) and
# to publish the combined dataset; not needed once it is cached on Drive.
KAGGLE_USERNAME = ""
KAGGLE_KEY = ""

IN_COLAB = "COLAB_GPU" in os.environ or os.path.exists("/content")
DRIVE_BASE = Path("/content/drive/MyDrive/yolo-waste-sorter")  # historical name kept: existing Drive checkpoints/runs resume from here

if IN_COLAB and not os.path.ismount("/content/drive"):
    from google.colab import drive

    drive.mount("/content/drive")
if IN_COLAB:
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)

# Export Kaggle creds to the env so the kaggle CLI (download + publish) finds them.
if KAGGLE_USERNAME and KAGGLE_KEY:
    os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
    os.environ["KAGGLE_KEY"] = KAGGLE_KEY

errors = []
if IN_COLAB:
    for name, value in {
        "GITHUB_USERNAME": GITHUB_USERNAME,
        "GITHUB_TOKEN": GITHUB_TOKEN,
        "REPO_URL": REPO_URL,
    }.items():
        if not value:
            errors.append(f"{name} is empty -- fill in the constants at the top of this cell")
assert not errors, "init failed:\n- " + "\n- ".join(errors)
print(f"init ok: IN_COLAB={IN_COLAB}, DRIVE_BASE={DRIVE_BASE if IN_COLAB else 'unused locally'}")


In [1]:
from pathlib import Path

pasta = Path("/content/TrashMonkey")

if (pasta / "src" / "trashmonkey").is_dir():
    print("SUCESSO: TrashMonkey clonado corretamente!")
    print("Pasta:", pasta)
else:
    print("ATENÇÃO: repositório ainda não encontrado.")

ATENÇÃO: repositório ainda não encontrado.


In [2]:
%cd /content
!git clone https://github.com/tr4m0ryp/TrashMonkey.git

/content
Cloning into 'TrashMonkey'...
remote: Enumerating objects: 1965, done.
remote: Counting objects: 100% (174/174), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 1965 (delta 97), reused 126 (delta 72), pack-reused 1791 (from 1)
Receiving objects: 100% (1965/1965), 609.58 KiB | 11.95 MiB/s, done.
Resolving deltas: 100% (1183/1183), done.


In [3]:
from pathlib import Path

pasta = Path("/content/TrashMonkey")

if pasta.is_dir():
    print("SUCESSO! TrashMonkey clonado.")
    print("\nArquivos encontrados:")
    for arquivo in pasta.iterdir():
        print("-", arquivo.name)
else:
    print("ERRO: pasta não encontrada.")

SUCESSO! TrashMonkey clonado.

Arquivos encontrados:
- pyproject.toml
- data
- Makefile
- tests
- .gitignore
- configs
- notebooks
- src
- README.md
- LICENSE
- .git
- models
- experiments


In [4]:
%cd /content/TrashMonkey

!pip install -e .

/content/TrashMonkey
Obtaining file:///content/TrashMonkey
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 MB 65.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 70.1 MB/s  0:00:00
  Building editable for trashmonkey (pyproject.toml) ... done
  Created wheel for trashmonkey: filename=trashmonkey-0.1.0-0.editable-py3-none-any.whl size=13623 sha256=6530e7c5783f8832aee8225a72268975ea4affcf12368a43554de2686e39dd97
  Stored in directory: /tmp/pip-ephem-wheel-cache-bgpintii/wheels/ea/3f/4b/5a308f19f17f356c5f0668d5cdbbe3d98fee6b0df6f8ea1c3b
Successfully built trashmonkey
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 5.0.0.93
    Uninstalling opencv-python-headless-5.0.0.93:
      Successfully uninstalled 

## 2. Clone / update repo

Idempotent: clones on the first run, fast-forward pulls afterwards. Local runs
locate the repo root instead of cloning.

In [ ]:
# Clone (Colab) or locate (local) the repo, then put src/ on sys.path.
import os
import subprocess
import sys
from pathlib import Path

if IN_COLAB:
    REPO_ROOT = Path("/content/TrashMonkey")
    # Build the authenticated URL in parts (the credential-in-URL literal
    # pattern would trip the repo's secret-scanning push gate).
    auth = GITHUB_USERNAME + ":" + GITHUB_TOKEN
    auth_url = REPO_URL.replace("https://", "https://" + auth + "@", 1)
    if (REPO_ROOT / ".git").is_dir():
        subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", auth_url, str(REPO_ROOT)], check=True)
else:
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    REPO_ROOT = next(
        path
        for path in candidates
        if (path / "pyproject.toml").is_file() and (path / "src" / "trashmonkey").is_dir()
    )

os.chdir(REPO_ROOT)
SRC_DIR = str(REPO_ROOT / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
print(f"repo root: {REPO_ROOT}")


## 2b. Install dependencies

Editable-install the package with the `boxing` extra so the runtime has
ultralytics + albumentations (train) and autodistill / grounding-dino / rembg
(the autobox stage of the dataset build). Colab only -- locally you already
have a venv.

**One-time restart:** installing pinned deps upgrades NumPy on disk, but Colab
already loaded the old NumPy into the kernel, so this cell **restarts the
runtime once** after installing. When it comes back, run **Runtime > Run all**
again -- the install is skipped (marker on `/content`) and everything proceeds.

In [ ]:
# Install the package + boxing extra (Colab). Locally this is skipped.
# Colab preloads an old numpy at kernel start; installing pinned deps upgrades
# numpy ON DISK, so the kernel must RESTART once to load the consistent set
# (else `import albumentations` hits a numpy ABI error). A token marker on
# /content makes this install + restart run once per dependency change: bump
# DEPS_TOKEN whenever the dependency set changes so a warm VM reinstalls.
import subprocess
import sys
from pathlib import Path

DEPS_TOKEN = "6-rembg-cpu"  # bump on any dependency change to force reinstall+restart

if IN_COLAB:
    marker = Path("/content/.trashmonkey_deps")
    current = marker.read_text().strip() if marker.is_file() else None
    if current != DEPS_TOKEN:
        # A prior run may have installed onnxruntime-gpu (CUDA-mismatched on Colab);
        # it shares the `onnxruntime` module with the CPU build and would shadow it,
        # so remove it before installing rembg[cpu]. No-op on a fresh VM.
        subprocess.run(
            [sys.executable, "-m", "pip", "uninstall", "-y", "onnxruntime-gpu"], check=False
        )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-e", ".[boxing]"],
            check=True,
            cwd=str(REPO_ROOT),
        )
        marker.write_text(DEPS_TOKEN)
        print("installed trashmonkey[boxing]; RESTARTING runtime so deps load "
              "cleanly -- when it finishes, run 'Runtime > Run all' AGAIN.")
        import IPython

        IPython.get_ipython().kernel.do_shutdown(restart=True)
        raise SystemExit("restarting runtime; re-run all cells")
    print(f"deps up to date (token {DEPS_TOKEN}) -- skipping reinstall/restart")
else:
    print("local run: assuming the project venv already has the dependencies")


## 2c. Preflight

Now that dependencies are installed, validate that the packages the pipeline
needs import cleanly and that ultralytics is new enough for the validated
custom-augmentation hook (`train(augmentations=[...])`, added in 8.3.226).

In [ ]:
# Preflight: validate imports + ultralytics version after the install cell.
import importlib

MIN_ULTRALYTICS = (8, 3, 226)  # train(augmentations=[...]) floor; training guards enforce it too
problems = []
for module_name in ["yaml", "PIL", "numpy", "albumentations", "torch", "ultralytics"]:
    try:
        importlib.import_module(module_name)
    except ImportError as exc:
        problems.append(f"package not importable: {module_name} ({exc})")

try:
    import ultralytics

    parsed = tuple(int(part) for part in ultralytics.__version__.split(".")[:3])
    if parsed < MIN_ULTRALYTICS:
        floor = ".".join(str(part) for part in MIN_ULTRALYTICS)
        problems.append(
            f"ultralytics {ultralytics.__version__} < {floor}: "
            "train(augmentations=[...]) needs the validated hook added in 8.3.226"
        )
except ImportError:
    pass  # already recorded above

assert not problems, "preflight failed:\n- " + "\n- ".join(problems)
print("preflight ok: all pipeline packages import; ultralytics >= 8.3.226")


## 3. Dataset: build-or-restore (Drive-cached)

Self-contained: on a **cache miss** the notebook runs the full data pipeline
in-process (download -> remap -> autobox -> qa -> dedup -> balance -> split) and
saves the result to Drive; on a **cache hit** it just restores the archive --
no rebuild. The cache key is a fingerprint of `datasets.yaml` plus the
dataset-relevant config fields (classes, seed, eval split), so a cap or
registry edit rebuilds automatically while an unchanged dataset is never
re-combined. Building needs Kaggle credentials (Init cell) the first time only.

In [ ]:
# Build the processed dataset on a cache miss, else restore it from Drive.
# restore_or_build fingerprints the inputs and picks: local (warm) -> restored
# (Drive archive matches) -> rebuilt (build + archive to Drive). save=IN_COLAB
# pushes the archive to Drive only on Colab; locally it just builds in place.
# Build progress: per-stage headers ([i/7] ...) plus a live percentage for the
# long autobox stage, so you can see it is working (train/eval print their own).
import os
from pathlib import Path

from trashmonkey.data.cache import restore_or_build
from trashmonkey.data.pipeline import build_dataset
from trashmonkey.utils.config import load_config
from trashmonkey.utils.progress import make_progress_printer, print_stage

cfg = load_config()
CONFIG_PATH = REPO_ROOT / "configs" / "config.yaml"
DATASETS_PATH = REPO_ROOT / "configs" / "datasets.yaml"
DATA_ROOT = REPO_ROOT / "data"
DATASET_YAML = REPO_ROOT / cfg.paths.processed / cfg.experiment.name / "dataset.yaml"
SPLIT_MANIFEST = REPO_ROOT / cfg.paths.interim / "pipeline" / "split.yaml"

# Persistent store: Google Drive on Colab, a local cache dir otherwise.
CACHE_BASE = (DRIVE_BASE / "data") if IN_COLAB else (DATA_ROOT / "cache")
ARCHIVE = CACHE_BASE / "processed.tar.gz"
FINGERPRINT = CACHE_BASE / "processed.fingerprint"
LOCAL_MARKER = DATA_ROOT / ".dataset.fingerprint"


def _build():
    # Kaggle creds are consulted ONLY here, on an actual (re)build.
    has_token = (Path.home() / ".kaggle" / "kaggle.json").is_file()
    has_env = bool(os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"))
    if not (has_token or has_env):
        raise RuntimeError(
            "dataset build needs Kaggle credentials: set KAGGLE_USERNAME/KAGGLE_KEY "
            "in the Init cell, or upload a token to ~/.kaggle/kaggle.json. "
            "(Not needed once the built dataset is cached on Drive.)"
        )
    # ack_review=True: unattended continue past the QA gate (documented mechanism).
    build_dataset(
        CONFIG_PATH,
        ack_review=True,
        on_stage=print_stage,
        on_progress=make_progress_printer(),
    )


decision = restore_or_build(
    config_path=CONFIG_PATH,
    datasets_path=DATASETS_PATH,
    data_root=DATA_ROOT,
    dataset_yaml=DATASET_YAML,
    archive_path=ARCHIVE,
    fingerprint_path=FINGERPRINT,
    local_marker=LOCAL_MARKER,
    build_fn=_build,
    save=IN_COLAB,
)
print(f"dataset {decision.status} (fingerprint {decision.fingerprint[:12]})")
print(f"dataset yaml:   {DATASET_YAML}")
print(f"archive:        {ARCHIVE if IN_COLAB else '(local build -- not archived)'}")
assert DATASET_YAML.is_file(), f"dataset yaml missing after {decision.status}: {DATASET_YAML}"
print(
    f"split manifest: {SPLIT_MANIFEST} "
    f"(exists={SPLIT_MANIFEST.is_file()}; the evaluate cell requires it)"
)


## 4. Smoke test

One tiny CPU epoch through the full train cycle. `train(smoke=True)`
synthesizes its own dataset when `data_yaml` is None and pins
device=cpu/imgsz=160/batch=2 internally.

In [ ]:
# Smoke test: full train cycle on CPU against a synthesized tiny dataset.
from trashmonkey.models.training import train
from trashmonkey.utils.config import load_config

cfg = load_config()
smoke_result = train(cfg, data_yaml=None, smoke=True)
assert smoke_result.best_pt.is_file(), f"smoke run produced no best.pt: {smoke_result.best_pt}"
print(f"smoke ok: best_pt={smoke_result.best_pt}")
print(f"smoke mAP50={smoke_result.metrics['map50']:.4f} run_dir={smoke_result.run_dir}")


## 5. Checkpoint / run detection

Lists prior run directories on the persistence root. The package's `train()`
exposes no resume parameter, so a fresh run is started either way (reported
as an API gap by task 013).

In [ ]:
# Prior-run detection on the persistence root: pick up an interrupted run.
# find_resumable() reads the epoch field inside each last.pt -- ultralytics
# strips finished runs to epoch=-1, so only genuinely interrupted runs match.
from trashmonkey.models.training import find_resumable

RUNS_ROOT = (DRIVE_BASE / "runs") if IN_COLAB else (REPO_ROOT / "experiments" / "runs")
RESUME_FROM = find_resumable(RUNS_ROOT)
if RESUME_FROM is not None:
    print(f"interrupted run detected -- training will resume from {RESUME_FROM}")
else:
    print(f"no interrupted run under {RUNS_ROOT}; training starts fresh")


## 5b. Runtime profile (auto-batch)

Detects whatever accelerator this session landed on and resolves `batch` /
`workers` from a static VRAM table (`training/autobatch.py`). Deterministic --
the same card always yields the same recipe, and the resolved integers are
logged in `runs.jsonl` -- so it is reproducible, unlike ultralytics
`batch=-1`. `batch` is capped at 64 (ultralytics' nominal batch size), so a
bigger GPU only changes throughput, not the optimisation. On a local CPU
session it falls back to the pinned `config.yaml` default (batch 16, workers 8).

In [ ]:
# Auto-batch: detect the accelerator once and resolve batch/workers for it.
# RUNTIME is reused by the train cell so detection happens exactly once.
from trashmonkey.models.training import detect_runtime

RUNTIME = detect_runtime()
print(f"runtime: {RUNTIME.summary()}")
if RUNTIME.device == "cpu":
    print("WARNING: no CUDA device -- a full 100-epoch run on CPU is impractical; "
          "this path is for the local smoke test only.")


## 6. Train

Config-driven T7 fine-tune. On Colab the run directory and `runs.jsonl` live
on Drive so checkpoints and the experiment log survive disconnects.

In [ ]:
# T7 fine-tune; checkpoints and the run log persist to Drive on Colab, so a
# disconnect loses nothing and the next "Run all" resumes via RESUME_FROM.
# apply_runtime() stamps the auto-detected batch/workers onto the config; the
# resolved values are logged in runs.jsonl like any other pinned arg.
from trashmonkey.models.training import DEFAULT_RUNS_JSONL, apply_runtime, train
from trashmonkey.utils.config import load_config

cfg, _ = apply_runtime(load_config(), RUNTIME)
print(f"training with batch={cfg.train.batch}, workers={cfg.train.workers} ({RUNTIME.summary()})")
RUNS_JSONL = (DRIVE_BASE / "experiments" / "runs.jsonl") if IN_COLAB else DEFAULT_RUNS_JSONL
run = train(
    cfg, data_yaml=DATASET_YAML, runs_jsonl=RUNS_JSONL, project=RUNS_ROOT, resume=RESUME_FROM
)
print(f"best_pt:    {run.best_pt}")
print(f"run_dir:    {run.run_dir}")
print(f"mAP50:      {run.metrics['map50']:.4f}")
for name, block in run.metrics["per_class"].items():
    print(f"  {name:<12} map50={block['map50']:.4f} recall={block['recall']:.4f}")
print(f"runs.jsonl: {RUNS_JSONL}")


## 7. Evaluate

Three-tier evaluation (VAL -> TEST-1 -> TEST-2 severity curve) plus the T7
escalation verdict that drives the yolo11n -> yolo11s decision.

In [ ]:
# Multi-tier evaluation + escalation verdict, now gated on the deployment-matched
# CLEAN tier (white-bg single-object) with config-driven floors; wild_test is a
# stress tier. clean/wild appear only when those splits exist in the dataset.
from trashmonkey.models.evaluation import evaluate
from trashmonkey.utils.config import load_config

cfg = load_config()
assert SPLIT_MANIFEST.is_file(), (
    f"split manifest missing: {SPLIT_MANIFEST}\n"
    "make repro writes it to data/interim/pipeline/split.yaml; include "
    "interim/pipeline in the uploaded archive (see the dataset cell)."
)
report = evaluate(cfg, run.best_pt, DATASET_YAML, SPLIT_MANIFEST)

header = f"{'class':<12}{'precision':>10}{'recall':>9}{'map50':>9}{'map50-95':>10}{'conf@p95':>10}"
tiers = [report.val, report.test1, *report.test2]
for optional in (report.clean, report.wild):  # present only when those splits exist
    if optional is not None:
        tiers.append(optional)
for tier in tiers:
    print(
        f"\n[{tier.tier}] split={tier.split} severity={tier.severity} "
        f"map50={tier.map50:.4f} map50-95={tier.map50_95:.4f}"
    )
    print(header)
    for name, cls_eval in sorted(tier.per_class.items()):
        conf = "never" if cls_eval.conf_at_p95 is None else f"{cls_eval.conf_at_p95:.3f}"
        print(
            f"{name:<12}{cls_eval.precision:>10.4f}{cls_eval.recall:>9.4f}"
            f"{cls_eval.map50:>9.4f}{cls_eval.map50_95:>10.4f}{conf:>10}"
        )

# Headline = the deployment-matched tier (clean_test if present, else VAL fallback).
if report.headline:
    h = report.headline
    print(
        f"\nHEADLINE [{h['tier']}]: deployment mAP50={h['map50']:.4f} "
        f"(secondary mAP50-95={h['map50_95']:.4f}) -- the number that matters"
    )

escalation = report.escalation
print("\n" + "=" * 76)
if escalation["passed"]:
    print("ESCALATION VERDICT: PASS -- meets the floors on the clean tier; stay on yolo11n")
else:
    print("ESCALATION VERDICT: FAIL -- clean-tier floors missed; consider yolo11n -> yolo11s")
print(
    f"overall mAP50 {escalation['overall_map50']:.4f} "
    f"(floor {escalation['thresholds']['overall_map50']})"
)
for name, block in escalation["per_class"].items():
    map50 = "absent" if block["map50"] is None else f"{block['map50']:.4f}"
    recall = "absent" if block["recall"] is None else f"{block['recall']:.4f}"
    print(f"  {name:<12} map50={map50} recall={recall} passed={block['passed']}")
print("=" * 76)


## 8. Plots

Supplied by task 014 (`trashmonkey/visualization/plots.py`); skipped
cleanly until it lands.

In [ ]:
# Plot stage (task 014): render every figure the run artifacts support.
from pathlib import Path

from trashmonkey.utils.config import load_config
from trashmonkey.visualization import plots

cfg = load_config()
PLOTS_DIR = (DRIVE_BASE / "plots") if IN_COLAB else (REPO_ROOT / "reports" / "figures")
PLOTS_DIR.mkdir(parents=True, exist_ok=True)


def _save(name, render):
    out = PLOTS_DIR / name
    render(out)
    print(f"saved {out}")


_save("dataset_composition.png", lambda o: plots.plot_dataset_composition(SPLIT_MANIFEST, save_path=o))
_save("tier_comparison.png", lambda o: plots.plot_tier_comparison(report, save_path=o))
_save("severity_curve.png", lambda o: plots.plot_severity_curve(report, save_path=o))
_save(
    "per_class_curves_val.png",
    lambda o: plots.plot_per_class_curves(report, Path(report.val.curves_path), save_path=o),
)

results_csv = run.run_dir / "results.csv"
if results_csv.is_file():
    _save("training_curves.png", lambda o: plots.plot_training_curves(results_csv, save_path=o))
else:
    print(f"skip training_curves: {results_csv} missing")

eval_dir = Path(report.detections_path).parent
wilderness = eval_dir / "wilderness_detections.jsonl"
if wilderness.is_file():
    _save(
        "openset_separation.png",
        lambda o: plots.plot_confidence_separation(
            Path(report.detections_path), wilderness, save_path=o,
            tau_frame=cfg.thresholds.tau_frame,
        ),
    )
else:
    print(f"skip openset_separation: {wilderness} missing (thresholds stage dumps it)")

sweep_csv = eval_dir.parent / "thresholds" / "sweep.csv"
if sweep_csv.is_file():
    _save("threshold_tradeoff.png", lambda o: plots.plot_threshold_tradeoff(sweep_csv, save_path=o))
else:
    print(f"skip threshold_tradeoff: {sweep_csv} missing (run models.thresholds first)")

sample = next(iter(sorted((DATASET_YAML.parent / "images" / "val").glob("*.jpg"))), None)
if sample is not None:
    _save("degradation_grid.png", lambda o: plots.plot_degradation_grid(sample, save_path=o, seed=cfg.seed))
else:
    print("skip degradation_grid: no val image found beside DATASET_YAML")

## 9. Summary

In [ ]:
# Run summary.
from pathlib import Path

from trashmonkey.models.evaluation import REPORT_FILENAME
from trashmonkey.utils.config import load_config

cfg = load_config()
print(
    f"experiment:  {cfg.experiment.name} (seed={cfg.seed}, base={cfg.model.base}, "
    f"imgsz={cfg.model.imgsz}, epochs={cfg.train.epochs}, batch={RUNTIME.batch}, "
    f"workers={RUNTIME.workers}, optimizer={cfg.train.optimizer}, lr0={cfg.train.lr0}, "
    f"cls_pw={cfg.train.cls_pw})"
)
print(f"runtime:     {RUNTIME.summary()}")
print(f"train mAP50: {run.metrics['map50']:.4f}  (in-distribution val)")
if report.headline:
    print(
        f"DEPLOYMENT:  {report.headline['map50']:.4f} mAP50 on [{report.headline['tier']}] "
        f"-- the white-bg single-object number that matters"
    )
for name, block in run.metrics["per_class"].items():
    print(f"  {name:<12} map50={block['map50']:.4f} recall={block['recall']:.4f}")
verdict = "PASS (stay on yolo11n)" if report.escalation["passed"] else "FAIL (consider yolo11s)"
print(f"escalation:  {verdict} (gated on the clean tier)")
print(f"run dir:     {run.run_dir}")
print(f"eval report: {Path(report.detections_path).parent / REPORT_FILENAME}")
print(f"plots dir:   {PLOTS_DIR}")
print(f"runs.jsonl:  {RUNS_JSONL}")
